# AURA beard-lab — LaMa 얼굴 fine-tune (Colab)

사용법 (v02부터 학습은 여기서):
1. 로컬에서 `train/make_colab_kit.sh` 실행 → `beard_lab_colab_kit.zip` 생성
2. zip을 구글 드라이브 최상위 `beard_lab/` 폴더에 업로드
3. 런타임 유형 = GPU(T4 이상) 선택 후 아래 셀을 순서대로 실행
4. 결과는 드라이브 `beard_lab/outputs/`에 저장됨 → 로컬로 내려받아 평가

⚠ **K-FACE(AI Hub) 데이터는 절대 업로드 금지** — 제출한 활용계획서에
"로컬 장비 보관, 클라우드 업로드 안 함"으로 서약했음. Colab에는
커먼즈/자체 촬영 데이터만 올린다. K-FACE 합류 학습은 로컬(MPS)로.

In [ ]:
!nvidia-smi
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -qo /content/drive/MyDrive/beard_lab/beard_lab_colab_kit.zip -d /content/lab
!ls /content/lab

In [ ]:
# base 가중치 다운로드 + sha256 고정 검증 (spike/lama_manifest.json과 동일 출처)
import hashlib, json, pathlib, urllib.request
man = json.load(open('/content/lab/spike/lama_manifest.json'))
dst = pathlib.Path('/content/lab/external/models/big-lama/big-lama.pt')
dst.parent.mkdir(parents=True, exist_ok=True)
if not dst.exists():
    for url in man['mirrors']:
        try:
            print('fetching', url)
            urllib.request.urlretrieve(url, dst)
            break
        except Exception as e:
            print('mirror failed:', e)
sha = hashlib.sha256(dst.read_bytes()).hexdigest()
assert sha == man['sha256'], f'sha mismatch: {sha}'
print('weights OK', sha[:12])

In [ ]:
# 학습 — 스텝·배치는 상황에 맞게. T4 기준 batch 8 권장, A100이면 16
!cd /content/lab && python train/finetune_lama.py \
    --data /content/lab/ft_data \
    --steps 3000 --batch 8 --device cuda --tag colab_v02

In [ ]:
# 결과를 드라이브로 복사 (가중치 + 샘플 시트 + run_manifest)
!mkdir -p /content/drive/MyDrive/beard_lab/outputs
!cp -r /content/lab/outputs/finetune/lama_face_colab_v02 /content/drive/MyDrive/beard_lab/outputs/
print('done — 로컬에서 eval_finetune.py로 평가하세요')